# NOC Prediction — GF Kit Only, GroupKFold(5)

In [ ]:
!pip install tabpfn==0.1.9 --force-reinstall -q

In [ ]:
import os, sys, io, zipfile, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

CSV_PATH  = '/kaggle/input/datasets/nguyenmanhhust/gf-kit-file/gf_groups.csv'
CKPT_PATH = '/kaggle/input/datasets/nguyenmanhhust/noc-class-chuan/noc_class_chuan'

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_FOLDS = 5
SEED    = 42
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')

In [ ]:
df     = pd.read_csv(CSV_PATH)
groups = df['group_id'].values
y      = df['target_noc'].values.astype(int) - 1
cols   = [c for c in df.columns if c not in ('group_id', 'target_noc')]
X      = df[cols].values.astype(np.float32)

print(f'Dataset: {X.shape}  unique groups: {len(np.unique(groups))}')
print(f'NOC dist: { {k+1: int((y==k).sum()) for k in range(5)} }')

In [ ]:
def balanced_resample(X, y, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    classes, counts = np.unique(y, return_counts=True)
    target = min(counts.max(), 3 * counts.min())
    idx = []
    for c, cnt in zip(classes, counts):
        c_idx = np.where(y == c)[0]
        idx.extend(rng.choice(c_idx, target, replace=(cnt < target)).tolist())
    arr = np.array(idx)[rng.permutation(len(idx))]
    return X[arr], y[arr]

## 1. XGBoost

In [ ]:
from xgboost import XGBClassifier

print('='*60, '\nXGBoost — GroupKFold(5)\n' + '='*60)
gkf = GroupKFold(n_splits=N_FOLDS)
xgb_macro, xgb_micro = [], []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    t0 = time.time()
    X_tr, y_tr = balanced_resample(X[tr_idx], y[tr_idx])
    X_va, y_va = X[va_idx], y[va_idx]
    model = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
        eval_metric='mlogloss', random_state=SEED, n_jobs=-1,
    )
    model.fit(X_tr, y_tr)
    preds = model.predict(X_va)
    macro = f1_score(y_va, preds, average='macro', zero_division=0)
    micro = f1_score(y_va, preds, average='micro', zero_division=0)
    xgb_macro.append(macro); xgb_micro.append(micro)
    print(f'  Fold {fold}: Macro={macro:.4f}  Micro={micro:.4f}  ({time.time()-t0:.0f}s)')

print(f'\nXGBoost  Macro-F1: {np.mean(xgb_macro):.4f} \u00b1 {np.std(xgb_macro):.4f}')
print(f'         Micro-F1: {np.mean(xgb_micro):.4f} \u00b1 {np.std(xgb_micro):.4f}')

## 2. CNN (1D Conv)

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, n_features, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 64,  3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.AdaptiveAvgPool1d(8),
            nn.Flatten(),
            nn.Linear(256*8, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),   nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )
    def forward(self, x): return self.net(x.unsqueeze(1))

print('='*60, f'\nCNN \u2014 GroupKFold(5)  device={DEVICE}\n' + '='*60)
gkf = GroupKFold(n_splits=N_FOLDS)
cnn_macro, cnn_micro = [], []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    t0 = time.time()
    X_tr, y_tr = balanced_resample(X[tr_idx], y[tr_idx])
    X_va, y_va = X[va_idx], y[va_idx]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)

    model = CNN1D(X.shape[1]).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    dl    = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                       batch_size=256, shuffle=True)
    Xv    = torch.FloatTensor(X_va).to(DEVICE)

    best_f1, no_imp, best_state = 0., 0, None
    for ep in range(1, 151):
        model.train()
        for xb, yb in dl:
            loss = F.cross_entropy(model(xb.to(DEVICE)), yb.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            preds = model(Xv).argmax(1).cpu().numpy()
        mf1 = f1_score(y_va, preds, average='macro', zero_division=0)
        if mf1 > best_f1: best_f1, no_imp, best_state = mf1, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else: no_imp += 1
        if no_imp >= 20: break

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        preds = model(Xv).argmax(1).cpu().numpy()
    macro = f1_score(y_va, preds, average='macro', zero_division=0)
    micro = f1_score(y_va, preds, average='micro', zero_division=0)
    cnn_macro.append(macro); cnn_micro.append(micro)
    print(f'  Fold {fold}: Macro={macro:.4f}  Micro={micro:.4f}  ({time.time()-t0:.0f}s)')
    print(classification_report(y_va, preds, target_names=['NOC=1','NOC=2','NOC=3','NOC=4','NOC=5'],
                                labels=[0,1,2,3,4], zero_division=0, digits=3))

print(f'\nCNN  Macro-F1: {np.mean(cnn_macro):.4f} \u00b1 {np.std(cnn_macro):.4f}')
print(f'     Micro-F1: {np.mean(cnn_micro):.4f} \u00b1 {np.std(cnn_micro):.4f}')

## 3. Forensic Transformer

In [ ]:
MARKERS = [
    'AMEL','CSF1PO','D10S1248','D12S391','D13S317','D16S539',
    'D18S51','D19S433','D1S1656','D21S11','D22S1045','D2S1338',
    'D2S441','D3S1358','D5S818','D6S1043','D7S820','D8S1179',
    'DYS391','FGA','SE33','TH01','TPOX','vWA',
]

class ForensicTransformer(nn.Module):
    def __init__(self, dims, d_model=64, nhead=4, num_layers=3, n_classes=5):
        super().__init__()
        self.projs = nn.ModuleList([nn.Linear(max(d,1), d_model) for d in dims])
        enc_layer  = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=256,
                                                dropout=0.1, batch_first=True)
        self.enc   = nn.TransformerEncoder(enc_layer, num_layers)
        self.head  = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, n_classes))
    def forward(self, splits):
        return self.head(self.enc(torch.stack([p(s) for p,s in zip(self.projs,splits)],dim=1)).mean(1))

def get_marker_indices(cols):
    mapping = {m: [] for m in MARKERS}
    for i, c in enumerate(cols):
        for m in MARKERS:
            if c.startswith(m+'_') or c==m: mapping[m].append(i); break
    indices = [mapping[m] for m in MARKERS]
    return indices, [max(len(idx),1) for idx in indices]

def to_splits(X_np, indices, device):
    return [torch.zeros(len(X_np),1,device=device) if not idx
            else torch.FloatTensor(X_np[:,idx]).to(device) for idx in indices]

print('='*60, f'\nForensicTransformer \u2014 GroupKFold(5)  device={DEVICE}\n' + '='*60)
indices, dims = get_marker_indices(cols)
gkf = GroupKFold(n_splits=N_FOLDS)
ft_macro, ft_micro = [], []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    t0 = time.time()
    X_tr, y_tr = balanced_resample(X[tr_idx], y[tr_idx])
    X_va, y_va = X[va_idx], y[va_idx]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr); X_va = scaler.transform(X_va)

    model = ForensicTransformer(dims).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    dl    = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                       batch_size=256, shuffle=True)

    best_f1, no_imp, best_state = 0., 0, None
    for ep in range(1, 151):
        model.train()
        for xb, yb in dl:
            loss = F.cross_entropy(model(to_splits(xb.numpy(),indices,DEVICE)), yb.to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            preds = model(to_splits(X_va,indices,DEVICE)).argmax(1).cpu().numpy()
        mf1 = f1_score(y_va, preds, average='macro', zero_division=0)
        if mf1 > best_f1: best_f1, no_imp, best_state = mf1, 0, {k:v.clone() for k,v in model.state_dict().items()}
        else: no_imp += 1
        if no_imp >= 20: break

    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        preds = model(to_splits(X_va,indices,DEVICE)).argmax(1).cpu().numpy()
    macro = f1_score(y_va, preds, average='macro', zero_division=0)
    micro = f1_score(y_va, preds, average='micro', zero_division=0)
    ft_macro.append(macro); ft_micro.append(micro)
    print(f'  Fold {fold}: Macro={macro:.4f}  Micro={micro:.4f}  ({time.time()-t0:.0f}s)')
    print(classification_report(y_va, preds, target_names=['NOC=1','NOC=2','NOC=3','NOC=4','NOC=5'],
                                labels=[0,1,2,3,4], zero_division=0, digits=3))

print(f'\nForensicTransformer  Macro-F1: {np.mean(ft_macro):.4f} \u00b1 {np.std(ft_macro):.4f}')
print(f'                     Micro-F1: {np.mean(ft_micro):.4f} \u00b1 {np.std(ft_micro):.4f}')

## 4. TabPFN

In [ ]:
from tabpfn import TabPFNClassifier

TABPFN_MAX_TRAIN = 3000

print('='*60)
print(f'TabPFN \u2014 GroupKFold(5)  (train capped at {TABPFN_MAX_TRAIN})')
print('='*60)

gkf = GroupKFold(n_splits=N_FOLDS)
tab_macro, tab_micro = [], []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    t0 = time.time()
    X_tr_full, y_tr_full = balanced_resample(X[tr_idx], y[tr_idx])
    X_va, y_va = X[va_idx], y[va_idx]

    if len(y_tr_full) > TABPFN_MAX_TRAIN:
        from sklearn.model_selection import StratifiedShuffleSplit
        sss = StratifiedShuffleSplit(n_splits=1, train_size=TABPFN_MAX_TRAIN, random_state=SEED+fold)
        sub_idx, _ = next(sss.split(X_tr_full, y_tr_full))
        X_tr, y_tr = X_tr_full[sub_idx], y_tr_full[sub_idx]
    else:
        X_tr, y_tr = X_tr_full, y_tr_full

    model = TabPFNClassifier(
        device='cuda' if torch.cuda.is_available() else 'cpu',
        N_ensemble_configurations=32,
    )
    model.fit(X_tr, y_tr)
    preds = model.predict(X_va)

    macro = f1_score(y_va, preds, average='macro', zero_division=0)
    micro = f1_score(y_va, preds, average='micro', zero_division=0)
    tab_macro.append(macro); tab_micro.append(micro)
    print(f'  Fold {fold}: Macro={macro:.4f}  Micro={micro:.4f}  train={len(y_tr)}  ({time.time()-t0:.0f}s)')
    print(classification_report(y_va, preds, target_names=['NOC=1','NOC=2','NOC=3','NOC=4','NOC=5'],
                                labels=[0,1,2,3,4], zero_division=0, digits=3))

print(f'\nTabPFN  Macro-F1: {np.mean(tab_macro):.4f} \u00b1 {np.std(tab_macro):.4f}')
print(f'        Micro-F1: {np.mean(tab_micro):.4f} \u00b1 {np.std(tab_micro):.4f}')

## 5. SetTransformer (finetune noc_class_chuan)

In [ ]:
import sys, io, zipfile
sys.path.insert(0, '/kaggle/input/datasets/nguyenmanhhust/set-transformer-noc-class')
from set_transformer import SetTransformerMixture

LOCI_ORDER = ['AMEL','CSF1PO','D1S1656','D2S1338','D2S441','D3S1358',
    'D5S818','D6S1043','D7S820','D8S1179','D10S1248','D12S391',
    'D13S317','D16S539','D18S51','D19S433','D21S11','D22S1045',
    'DYS391','FGA','SE33','TH01','TPOX','vWA']
L2I = {l: i for i, l in enumerate(LOCI_ORDER)}
N_LOCI = len(LOCI_ORDER)
MAX_SEQ, D_MODEL = 120, 128

def load_pretrained(path):
    buf = io.BytesIO()
    with zipfile.ZipFile(path,'r') as zf:
        with zipfile.ZipFile(buf,'w') as out:
            for name in zf.namelist(): out.writestr(name, zf.read(name))
    buf.seek(0)
    return torch.load(buf, map_location='cpu', weights_only=True)

def build_tokens_st(X_np, cols):
    peak_spec = []
    for ci, col in enumerate(cols):
        for locus in LOCI_ORDER:
            if col.startswith(locus+'_'):
                try: peak_spec.append((L2I[locus], float(col[len(locus)+1:]), ci))
                except ValueError: pass
                break
        if col == 'AMEL_X': peak_spec.append((L2I['AMEL'], 1.0, ci))
        if col == 'AMEL_Y': peak_spec.append((L2I['AMEL'], 2.0, ci))
    N = len(X_np)
    lidxs = np.array([p[0] for p in peak_spec], dtype=np.float32)
    avals = np.array([p[1] for p in peak_spec], dtype=np.float32)
    H     = X_np[:, [p[2] for p in peak_spec]].astype(np.float32)
    logH  = np.log1p(H)
    tokens = np.zeros((N, MAX_SEQ, 3), dtype=np.float32)
    masks  = np.zeros((N, MAX_SEQ), dtype=bool)
    for i in range(N):
        present = np.where(H[i]>0)[0][:MAX_SEQ]; n = len(present)
        tokens[i,:n,0]=lidxs[present]; tokens[i,:n,1]=avals[present]; tokens[i,:n,2]=logH[i,present]
        masks[i,:n]=True
    return tokens, masks

def build_auxiliary(tokens, masks):
    B = tokens.shape[0]
    aux = np.zeros((B, N_LOCI + 1), dtype=np.float32)
    for i in range(B):
        n = int(masks[i].sum())
        for lid in tokens[i, :n, 0].astype(int): aux[i, lid] += 1
        aux[i, N_LOCI] = n
    return aux

class NOCDataset(Dataset):
    def __init__(self, tok, msk, aux, lab):
        self.tok = torch.from_numpy(tok); self.msk = torch.from_numpy(msk)
        self.aux = torch.from_numpy(aux); self.lab = torch.tensor(lab, dtype=torch.long)
    def __len__(self): return len(self.lab)
    def __getitem__(self, i): return self.tok[i], self.msk[i], self.aux[i], self.lab[i]

class FineTunedNOC(nn.Module):
    def __init__(self, pretrained_sd, aux_dim=25):
        super().__init__()
        self.backbone = SetTransformerMixture(
            n_loci=24, d_locus=16, d_model=D_MODEL, n_heads=4, n_isab=2, m_inducing=32,
            n_classes=45, n_noc=6, dropout=0.1, cls_decoder='pooled', encoder='isab',
            n_token_feats=3, num_embed='raw',
        )
        ignore = {'cls_head','reject_head','noc_head'}
        compat = {k:v for k,v in pretrained_sd.items() if not any(k.startswith(p) for p in ignore)}
        miss, _ = self.backbone.load_state_dict(compat, strict=False)
        print(f'    Backbone: {len(compat)} keys  missing={len(miss)}')
        head_in = D_MODEL + aux_dim
        self.noc_head = nn.Sequential(
            nn.LayerNorm(head_in),
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(128, 5),
        )
    def forward(self, tok, msk, aux):
        return self.noc_head(torch.cat([self.backbone.encode(tok, msk), aux], dim=-1))

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()

def oversample_full(tok, msk, aux, lab, rng):
    classes, counts = np.unique(lab, return_counts=True); target = counts.max(); idx = []
    for c, cnt in zip(classes, counts):
        c_idx = np.where(lab == c)[0]
        idx.extend(rng.choice(c_idx, target, replace=(cnt < target)).tolist())
    arr = np.array(idx)[rng.permutation(len(idx))]
    return tok[arr], msk[arr], aux[arr], lab[arr]

print('='*60)
print(f'SetTransformer (focal+aux+oversample) \u2014 GroupKFold(5)  device={DEVICE}')
print('='*60)

pretrained_sd = load_pretrained(CKPT_PATH)
print(f'Pretrained: {len(pretrained_sd)} tensors')

tokens_st, masks_st = build_tokens_st(X, cols)
valid = masks_st.sum(1) > 0
tokens_st, masks_st = tokens_st[valid], masks_st[valid]
y_st, g_st = y[valid], groups[valid]
aux = build_auxiliary(tokens_st, masks_st)
print(f'Dataset: {tokens_st.shape[0]} samples  peaks/sample: {masks_st.sum(1).mean():.1f}')

class_counts = np.bincount(y_st, minlength=5).astype(np.float32)
class_weights = 1.0 / class_counts
class_weights /= class_weights.min()
cw_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

gkf = GroupKFold(n_splits=N_FOLDS)
sett_macro, sett_micro = [], []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(tokens_st, y_st, g_st), 1):
    t0 = time.time()
    rng = np.random.default_rng(SEED + fold); torch.manual_seed(SEED + fold)
    print(f'\n  Fold {fold}/{N_FOLDS}  train={len(tr_idx)}  val={len(va_idx)}')

    tok_tr, msk_tr, aux_tr, lab_tr = oversample_full(
        tokens_st[tr_idx], masks_st[tr_idx], aux[tr_idx], y_st[tr_idx], rng)
    tr_dl = DataLoader(NOCDataset(tok_tr, msk_tr, aux_tr, lab_tr), 256, shuffle=True)
    va_dl = DataLoader(NOCDataset(tokens_st[va_idx], masks_st[va_idx], aux[va_idx], y_st[va_idx]),
                       512, shuffle=False)

    model = FineTunedNOC(pretrained_sd, aux_dim=N_LOCI+1).to(DEVICE)
    criterion = FocalLoss(weight=cw_tensor, gamma=2.0)
    opt = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': 2e-4},
        {'params': model.noc_head.parameters(),  'lr': 5e-4},
    ], weight_decay=1e-4)
    warmup_ep, total_ep = 5, 80
    def lr_lambda(ep):
        if ep < warmup_ep: return (ep + 1) / warmup_ep
        return 0.5 * (1 + np.cos(np.pi * (ep - warmup_ep) / (total_ep - warmup_ep)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    best_f1, no_imp, best_state = 0., 0, None
    for ep in range(1, total_ep + 1):
        model.train()
        for tb, mb, ab, lb in tr_dl:
            logits = model(tb.to(DEVICE), mb.to(DEVICE), ab.to(DEVICE))
            loss = criterion(logits, lb.to(DEVICE))
            if not torch.isnan(loss):
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        sched.step()
        model.eval(); pl, tl = [], []
        with torch.no_grad():
            for tb, mb, ab, lb in va_dl:
                pl.append(model(tb.to(DEVICE), mb.to(DEVICE), ab.to(DEVICE)).argmax(1).cpu().numpy())
                tl.append(lb.numpy())
        mf1 = f1_score(np.concatenate(tl), np.concatenate(pl), average='macro', zero_division=0)
        if mf1 > best_f1:
            best_f1, no_imp = mf1, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else: no_imp += 1
        if ep % 10 == 0 or ep <= 3:
            print(f'    ep{ep:3d}  val_macro={mf1:.4f}  best={best_f1:.4f}')
        if no_imp >= 15:
            print(f'    early stop @ ep{ep}  best={best_f1:.4f}'); break

    model.load_state_dict(best_state); model.eval(); pl, tl = [], []
    with torch.no_grad():
        for tb, mb, ab, lb in va_dl:
            pl.append(model(tb.to(DEVICE), mb.to(DEVICE), ab.to(DEVICE)).argmax(1).cpu().numpy())
            tl.append(lb.numpy())
    preds = np.concatenate(pl); trues = np.concatenate(tl)
    macro = f1_score(trues, preds, average='macro', zero_division=0)
    micro = f1_score(trues, preds, average='micro', zero_division=0)
    sett_macro.append(macro); sett_micro.append(micro)
    print(f'  Fold {fold}: Macro={macro:.4f}  Micro={micro:.4f}  ({time.time()-t0:.0f}s)')
    print(classification_report(trues, preds, target_names=['NOC=1','NOC=2','NOC=3','NOC=4','NOC=5'],
                                labels=[0,1,2,3,4], zero_division=0, digits=3))

print(f'\nSetTransformer (noc_class_chuan)  Macro-F1: {np.mean(sett_macro):.4f} \u00b1 {np.std(sett_macro):.4f}')
print(f'                                  Micro-F1: {np.mean(sett_micro):.4f} \u00b1 {np.std(sett_micro):.4f}')

## 6. SetTransformer v4 (noc_10mb finetune, 8-dim tokens)

In [ ]:
sys.path.insert(0, '/kaggle/input/datasets/mashwoo/set-transformer-10mb')
DATA_CSV_V4 = '/kaggle/input/datasets/nguyenmanhhust/gf-kit-file/gf_groups.csv'
CKPT_V4     = '/kaggle/input/datasets/nguyenmanhhust/noc-10mb/noc_10mb'

from set_transformer_10mb import SetTransformerMixture as SetTransformerMixture10mb

ALLELE_MAP_V4 = {'X': -2.0, 'Y': -1.0}
SKIP_V4       = {'TPH', 'PeakCount', 'MaxHeight'}

df_v4        = pd.read_csv(DATA_CSV_V4)
locus_cols_v4 = [c for c in df_v4.columns
                 if c not in ('target_noc', 'group_id')
                 and c.rsplit('_', 1)[1] not in SKIP_V4]
noc_v4        = df_v4['target_noc'].values
group_ids_v4  = df_v4['group_id'].values
vals_v4       = df_v4[locus_cols_v4].values.astype(np.float32)

loci_names_v4 = sorted(set(c.rsplit('_', 1)[0] for c in locus_cols_v4))
locus2idx_v4  = {l: i for i, l in enumerate(loci_names_v4)}
col_info_v4   = []
for col in locus_cols_v4:
    ln, als = col.rsplit('_', 1)
    li = locus2idx_v4.get(ln, 0)
    av = ALLELE_MAP_V4.get(als, None)
    if av is None:
        try: av = float(als)
        except: av = 0.0
    col_info_v4.append((li, av))

def build_tokens_v4(row_vals, col_info, S=200):
    N = len(row_vals)
    tokens = np.zeros((N, S, 3), dtype=np.float32)
    masks  = np.zeros((N, S),    dtype=np.float32)
    for i in range(N):
        pos = 0
        for j, (li, av) in enumerate(col_info):
            h = row_vals[i, j]
            if h <= 0 or np.isnan(h): continue
            if pos < S:
                tokens[i, pos, 0] = li; tokens[i, pos, 1] = av
                tokens[i, pos, 2] = np.log1p(h); masks[i, pos] = 1.0; pos += 1
    return tokens, masks

def enrich_v4(tokens, masks):
    N, S, _ = tokens.shape
    e = np.zeros((N, S, 8), dtype=np.float32); e[:, :, :3] = tokens
    for i in range(N):
        vi = np.where(masks[i] > 0)[0]
        if len(vi) == 0: continue
        loci  = tokens[i, vi, 0].astype(int); raw_h = np.expm1(tokens[i, vi, 2])
        h_max = raw_h.max() + 1e-9; lp = {}
        for j, idx in enumerate(vi): lp.setdefault(int(loci[j]), []).append((idx, tokens[i, idx, 1], raw_h[j]))
        for l, peaks in lp.items():
            n_l = len(peaks); lsum = sum(p[2] for p in peaks) + 1e-9
            ps  = sorted(peaks, key=lambda x: -x[2]); ah = {round(p[1], 1): p[2] for p in peaks}
            for rank, (idx, av, h) in enumerate(ps):
                ph = ah.get(round(av + 1.0, 1), 0.0)
                e[i, idx, 3] = h / lsum
                e[i, idx, 4] = h / (ph + 1e-9) if ph > 0 else 0.0
                e[i, idx, 5] = rank / max(n_l - 1, 1)
                e[i, idx, 6] = n_l / 12.0
                e[i, idx, 7] = h / h_max
    return e

tok3_v4, msk_v4 = build_tokens_v4(vals_v4, col_info_v4)
max_len_v4 = int(msk_v4.sum(1).max()) + 2
tok8_v4    = enrich_v4(tok3_v4[:, :max_len_v4, :], msk_v4[:, :max_len_v4])
msk_v4     = msk_v4[:, :max_len_v4]

ckpt_v4      = torch.load(CKPT_V4, map_location='cpu', weights_only=False)
owner_lut_v4 = ckpt_v4.get('owner_lut', None)

class NOCFinetuneV4(nn.Module):
    def __init__(self, pretrain_sd=None, owner_lut=None, d_model=128, n_noc=5, dropout=0.1):
        super().__init__()
        use_pt = pretrain_sd is not None
        self.backbone = SetTransformerMixture10mb(
            n_loci=24, d_locus=16, d_model=d_model, n_heads=4, n_isab=2, m_inducing=32,
            n_classes=45, n_noc=5, dropout=dropout, cls_decoder='pooled',
            n_token_feats=8, encoder='isab++', num_embed='periodic',
            n_freq=8, d_num_emb=8, periodic_sigma=0.3, nc_attn='mab0',
            feas_filter=use_pt, set_of_set=use_pt,
            owner_lut=owner_lut if use_pt else None,
            aux_heads=use_pt, noc_head_v2=use_pt,
        )
        if use_pt:
            missing, unexpected = self.backbone.load_state_dict(pretrain_sd, strict=False)
            print(f'  Pretrain: loaded {len(pretrain_sd)-len(unexpected)}/{len(pretrain_sd)} tensors')
        self.noc_head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(d_model, 64),
            nn.ReLU(True), nn.Dropout(dropout), nn.Linear(64, n_noc),
        )
    def forward(self, tokens, mask):
        mb = mask.bool() if mask.dtype != torch.bool else mask
        return self.noc_head(self.backbone.encode(tokens, mb))

class FocalLossV4(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()

def oversample_v4(indices, labels):
    from collections import Counter
    counts = Counter(labels[indices].tolist()); target = max(counts.values()); out = []
    for cls, cnt in counts.items():
        idx = indices[labels[indices] == cls]
        if cnt < target:
            reps = target // cnt; extra = target % cnt
            idx  = np.concatenate([np.tile(idx, reps), np.random.choice(idx, extra, replace=False)])
        out.append(idx)
    return np.concatenate(out)

def train_fold_v4(model, tok_tr, msk_tr, lbl_tr, tok_val, msk_val, lbl_val,
                  device, epochs=60, lr=3e-4, bs=64, patience=12):
    unique, counts = np.unique(lbl_tr, return_counts=True)
    total = counts.sum()
    cw = torch.tensor([total / (len(unique) * c) for c in counts], dtype=torch.float32).to(device)
    criterion = FocalLossV4(weight=cw)

    tr_idx = oversample_v4(np.arange(len(lbl_tr)), lbl_tr); np.random.shuffle(tr_idx)
    dl = DataLoader(TensorDataset(torch.tensor(tok_tr[tr_idx], dtype=torch.float32),
                                   torch.tensor(msk_tr[tr_idx], dtype=torch.float32),
                                   torch.tensor(lbl_tr[tr_idx], dtype=torch.long)),
                    batch_size=bs, shuffle=True, drop_last=True)
    Xv = torch.tensor(tok_val, dtype=torch.float32).to(device)
    Mv = torch.tensor(msk_val, dtype=torch.float32).to(device)
    Yv = torch.tensor(lbl_val, dtype=torch.long)

    for p in model.backbone.parameters(): p.requires_grad = False
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr*3, weight_decay=1e-4)
    model.train()
    for _ in range(5):
        for xb, mb, yb in dl:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            opt.zero_grad(); criterion(model(xb, mb), yb).backward(); opt.step()
    for p in model.backbone.parameters(): p.requires_grad = True

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    best_macro, best_state, wait = 0.0, None, 0

    for ep in range(epochs):
        model.train(); total_loss = 0
        for xb, mb, yb in dl:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb, mb), yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); total_loss += loss.item()
        sched.step()
        model.eval()
        with torch.no_grad():
            preds = torch.cat([model(Xv[i:i+bs], Mv[i:i+bs]).argmax(1)
                               for i in range(0, len(Xv), bs)]).cpu().numpy()
        macro = f1_score(Yv.numpy(), preds, average='macro')
        if macro > best_macro:
            best_macro = macro; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; wait = 0
        else: wait += 1
        if (ep + 1) % 10 == 0 or wait == 0:
            print(f'    Ep {ep+1:3d}: loss={total_loss/len(dl):.4f}  val_macro={macro:.4f}  best={best_macro:.4f}')
        if wait >= patience: print(f'    Early stop ep {ep+1}'); break

    model.load_state_dict(best_state)
    return best_macro

labels_v4 = noc_v4 - 1
gkf_v4 = GroupKFold(n_splits=5)
fold_macros_v4 = []

print('\n' + '='*60)
print('  SetTransformer v4 (noc_10mb) \u2014 GroupKFold(5) GF only')
print('='*60)

for fold, (tr_idx, val_idx) in enumerate(gkf_v4.split(tok8_v4, labels_v4, groups=group_ids_v4), 1):
    print(f'\n  Fold {fold}:')
    t0    = time.time()
    model = NOCFinetuneV4(pretrain_sd=ckpt_v4, owner_lut=owner_lut_v4).to(DEVICE)
    macro = train_fold_v4(model,
                          tok8_v4[tr_idx], msk_v4[tr_idx], labels_v4[tr_idx],
                          tok8_v4[val_idx], msk_v4[val_idx], labels_v4[val_idx],
                          device=DEVICE, epochs=60, lr=3e-4, bs=64, patience=12)

    model.eval()
    Xv = torch.tensor(tok8_v4[val_idx], dtype=torch.float32).to(DEVICE)
    Mv = torch.tensor(msk_v4[val_idx],  dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        preds = torch.cat([model(Xv[i:i+64], Mv[i:i+64]).argmax(1)
                           for i in range(0, len(Xv), 64)]).cpu().numpy()
    print(f'\n  Fold {fold}: Macro={macro:.4f}  ({time.time()-t0:.0f}s)')
    print(classification_report(labels_v4[val_idx], preds,
          target_names=[f'NOC={k}' for k in range(1, 6)], digits=3))
    fold_macros_v4.append(macro)

print(f'\n  SetTransformer v4  Macro-F1: {np.mean(fold_macros_v4):.4f} \u00b1 {np.std(fold_macros_v4):.4f}')

## Summary

In [ ]:
print('='*68)
print('  NOC PREDICTION \u2014 GF Kit Only, GroupKFold(5) \u2014 FINAL RESULTS')
print('='*68)
print(f'  {"Model":<35} {"Macro-F1":>14}  {"Micro-F1":>14}')
print('-'*68)
for name, ml, mli in [
    ('XGBoost',                          xgb_macro,      xgb_micro),
    ('CNN (1D Conv)',                     cnn_macro,      cnn_micro),
    ('ForensicTransformer',               ft_macro,       ft_micro),
    ('TabPFN',                            tab_macro,      tab_micro),
    ('SetTransformer (noc_class_chuan)',  sett_macro,     sett_micro),
    ('SetTransformer v4 (noc_10mb)',      fold_macros_v4, [0.0]*5),
]:
    macro_s = f'{np.mean(ml):.4f}\u00b1{np.std(ml):.4f}'
    micro_s = f'{np.mean(mli):.4f}\u00b1{np.std(mli):.4f}' if any(m > 0 for m in mli) else 'N/A'
    print(f'  {name:<35} {macro_s:>14}  {micro_s:>14}')
print('='*68)

## 7. Save Per-Fold Scores for Paired t-Tests

Export all per-fold Macro F1 scores to JSON so `paper_kaggle.ipynb` can run
paired t-tests between DS-ST and each baseline.

In [ ]:
import json
from scipy import stats

baseline_scores = {
    'XGBoost':              xgb_macro,
    'CNN':                  cnn_macro,
    'ForensicTransformer':  ft_macro,
    'TabPFN':               tab_macro,
}

with open('baseline_perfold.json', 'w') as f:
    json.dump(baseline_scores, f, indent=2)

print('Saved baseline_perfold.json')
print()
print('Per-fold scores:')
for name, scores in baseline_scores.items():
    print(f'  {name:<25} {["%.4f" % s for s in scores]}')
    print(f'  {"":25} mean={np.mean(scores):.4f} +/- {np.std(scores):.4f}')